In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

print("Generating synthetic EV battery degradation data...")
np.random.seed(42)
cycles = np.arange(1, 201)

data = {
    'Cycle': cycles,
    'Voltage_measured': 4.2 - (cycles * 0.002) + np.random.normal(0, 0.01, 200),
    'Current_measured': 1.5 + np.random.normal(0, 0.05, 200),
    'Temperature_measured': 25 + (cycles * 0.05) + np.random.normal(0, 0.2, 200),
    'Capacity': 2.0 - (cycles * 0.003) + np.random.normal(0, 0.005, 200)
}
df = pd.DataFrame(data)
nominal_capacity = 2.0
df['SoH'] = df['Capacity'] / nominal_capacity
df = df.sort_values(by='Cycle').reset_index(drop=True)

feature_columns = ['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Capacity']
target_column = 'SoH'

X = df[feature_columns]
y = df[target_column]

print("Starting TimeSeriesSplit Cross-Validation...\n")
tscv = TimeSeriesSplit(n_splits=5)

rf_rmse_scores, rf_accuracy_scores = [], []
xgb_rmse_scores, xgb_accuracy_scores = [], []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # --- Random Forest ---
    
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
    # Calculate percentage accuracy
    
    rf_mape = mean_absolute_percentage_error(y_test, rf_preds)
    rf_accuracy = (1 - rf_mape) * 100
    
    rf_rmse_scores.append(rf_rmse)
    rf_accuracy_scores.append(rf_accuracy)
    
    # --- XGBoost ---
    
    xgb = XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.05)
    xgb.fit(X_train, y_train)
    xgb_preds = xgb.predict(X_test)
    
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
    # Calculate percentage accuracy
    
    xgb_mape = mean_absolute_percentage_error(y_test, xgb_preds)
    xgb_accuracy = (1 - xgb_mape) * 100
    
    xgb_rmse_scores.append(xgb_rmse)
    xgb_accuracy_scores.append(xgb_accuracy)
    
    print(f"Fold {fold+1}:")
    print(f"  [RF]  RMSE: {rf_rmse:.4f} | Accuracy: {rf_accuracy:.2f}%")
    print(f"  [XGB] RMSE: {xgb_rmse:.4f} | Accuracy: {xgb_accuracy:.2f}%")
    print("-" * 45)


#  RESULTS

print("\n=== FINAL CROSS-VALIDATION PERFORMANCE ===")
summary_data = {
    'Model': ['Random Forest', 'XGBoost'],
    'Mean RMSE': [np.mean(rf_rmse_scores), np.mean(xgb_rmse_scores)],
    'Mean Accuracy (%)': [f"{np.mean(rf_accuracy_scores):.2f}%", f"{np.mean(xgb_accuracy_scores):.2f}%"]
}
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))


Generating synthetic EV battery degradation data...
Starting TimeSeriesSplit Cross-Validation...

Fold 1:
  [RF]  RMSE: 0.0292 | Accuracy: 97.19%
  [XGB] RMSE: 0.0292 | Accuracy: 97.17%
---------------------------------------------
Fold 2:
  [RF]  RMSE: 0.0318 | Accuracy: 96.70%
  [XGB] RMSE: 0.0324 | Accuracy: 96.64%
---------------------------------------------
Fold 3:
  [RF]  RMSE: 0.0302 | Accuracy: 96.74%
  [XGB] RMSE: 0.0318 | Accuracy: 96.53%
---------------------------------------------
Fold 4:
  [RF]  RMSE: 0.0290 | Accuracy: 96.71%
  [XGB] RMSE: 0.0305 | Accuracy: 96.50%
---------------------------------------------
Fold 5:
  [RF]  RMSE: 0.0309 | Accuracy: 96.21%
  [XGB] RMSE: 0.0325 | Accuracy: 95.97%
---------------------------------------------

=== FINAL CROSS-VALIDATION PERFORMANCE ===
        Model  Mean RMSE Mean Accuracy (%)
Random Forest   0.030228            96.71%
      XGBoost   0.031266            96.56%
